# EDA on log2FC and synergy data

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Data root configs.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  l2fc_dir = str(data_dir / "dge")
  cfu_dir = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    get_l2fc_and_cfu_data,
    attach_synergy_metadata,
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Log2FC data alone
l2fc_df = attach_synergy_metadata(get_l2fc_and_cfu_data(l2fc_dir, cfu_dir, time_matched = True))

# Bliss score and simple interaction score
synergy_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
synergy_df = synergy_df.dropna(axis = 1)

# Load annotations
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)

## PCA on log2FC data

Run PCA and color by drug.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Sample list
drug_list = ["CEF", "CIP", "CEF+CIP"]

# Filter to genes
X = l2fc_df[l2fc_df["drug_id"].isin(drug_list)]
X = X.iloc[:, X.columns.str.contains("SP")]
X = X.dropna(axis = 1)

# Run PCA
pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components = 2))
])
pca_results = pd.DataFrame(pca.fit_transform(X), index = X.index)
pca_results.columns = ["PC1", "PC2"]
pca_results = pd.merge(pca_results, l2fc_df, left_index = True, right_index = True, how = "left")

# Plot
sns.scatterplot(data = pca_results, x = "PC1", y = "PC2", hue = "drug_id")

## Heatmaps for log2FC data

All genes.

Cell wall genes (CEF, VNC).

Replication genes (CIP).

Transcription genes (RIF).

## Synergy data

In [ ]:
sns.stripplot(synergy_df, x = "drug_id", y = "synergy_score", hue = "drug_id")
plt.title("Synergy score distribution")
plt.xlabel("Drug combination")
plt.ylabel("EOB synergy score")